In [ ]:
from pathlib import Path
import pandas as pd


project_root = Path.cwd().parent
file_path = project_root / "data" / "processed" / "retail_store_inventory_clean.csv"

df = pd.read_csv(file_path, parse_dates=["Date"])

df.head()

In [ ]:
df[
    [
        "Inventory Level",
        "Units Sold",
        "Units Ordered",
        "Demand Forecast",
        "Price",
        "Discount",
        "Estimated Sales"
    ]
].describe().round(2)

In [4]:
ventas_categoria = (
    df.groupby("Category")
    .agg(
        unidades_vendidas=("Units Sold", "sum"),
        ventas_estimadas=("Estimated Sales", "sum")
    )
    .sort_values("ventas_estimadas", ascending=False)
    .round(2)
)

ventas_categoria

,unidades_vendidas,ventas_estimadas
Category,,
Furniture,2025017,1.002308e+08
Groceries,2000482,9.994897e+07
Toys,1990485,9.872922e+07
Clothing,1999166,9.867678e+07
Electronics,1960432,9.738559e+07


In [5]:
ventas_region = (
    df.groupby("Region")
    .agg(
        unidades_vendidas=("Units Sold", "sum"),
        ventas_estimadas=("Estimated Sales", "sum")
    )
    .sort_values("ventas_estimadas", ascending=False)
    .round(2)
)

ventas_region

,unidades_vendidas,ventas_estimadas
Region,,
East,2511265,1.247922e+08
South,2507799,1.245674e+08
North,2484966,1.235437e+08
West,2471552,1.220681e+08


In [6]:
ventas_tienda = (
    df.groupby("Store ID")
    .agg(
        unidades_vendidas=("Units Sold", "sum"),
        ventas_estimadas=("Estimated Sales", "sum")
    )
    .sort_values("ventas_estimadas", ascending=False)
    .round(2)
)

ventas_tienda

,unidades_vendidas,ventas_estimadas
Store ID,,
S005,2010176,1.005036e+08
S003,2022696,1.000344e+08
S002,1987715,9.895000e+07
S004,1979245,9.805714e+07
S001,1975750,9.742619e+07


In [ ]:
ventas_producto = (
    df.groupby("Product ID")
    .agg(
        unidades_vendidas=("Units Sold", "sum"),
        ventas_estimadas=("Estimated Sales", "sum")
    )
    .sort_values("ventas_estimadas", ascending=False)
    .round(2)
)

ventas_producto


In [8]:
ventas_mensuales = (
    df.groupby(df["Date"].dt.to_period("M"))
    .agg(
        unidades_vendidas=("Units Sold", "sum"),
        ventas_estimadas=("Estimated Sales", "sum")
    )
)

ventas_mensuales

,unidades_vendidas,ventas_estimadas
Date,,
2022-01,419938,2.091953e+07
2022-02,391052,1.906103e+07
2022-03,426073,2.152268e+07
2022-04,407380,2.031440e+07
2022-05,414799,2.054256e+07
2022-06,415509,2.098179e+07
2022-07,426628,2.129391e+07
2022-08,424916,2.105528e+07
2022-09,411610,2.062891e+07


In [9]:
inventario_demanda = (
    df[
        ["Inventory Level", "Demand Forecast", "Units Sold"]
    ]
    .mean()
    .round(2)
)

inventario_demanda

Inventory Level    274.47
Demand Forecast    141.53
Units Sold         136.46
dtype: float64

In [10]:
df["Brecha Inventario Demanda"] = (
    df["Inventory Level"] - df["Demand Forecast"]
)

df["Brecha Inventario Demanda"].describe().round(2)

count    73100.00
mean       132.94
std        110.03
min        -19.97
25%         44.44
50%        104.78
75%        200.08
max        499.00
Name: Brecha Inventario Demanda, dtype: float64

In [11]:
casos_bajo_forecast = (
    df["Brecha Inventario Demanda"] < 0
).sum()

casos_bajo_forecast

np.int64(2585)

In [12]:
porcentaje_bajo_forecast = (
    casos_bajo_forecast / len(df) * 100
)

round(porcentaje_bajo_forecast, 2)

np.float64(3.54)

In [13]:
riesgo_categoria = (
    df[df["Brecha Inventario Demanda"] < 0]
    .groupby("Category")
    .size()
    .sort_values(ascending=False)
)

riesgo_categoria

Category
Groceries      535
Furniture      524
Clothing       510
Electronics    508
Toys           508
dtype: int64

In [14]:
deficit = (
    df[df["Brecha Inventario Demanda"] < 0]
    ["Brecha Inventario Demanda"]
)

deficit.describe().round(2)


count    2585.00
mean       -6.69
std         4.84
min       -19.97
25%       -10.08
50%        -5.80
75%        -2.62
max        -0.01
Name: Brecha Inventario Demanda, dtype: float64

In [15]:
riesgo = df[df["Brecha Inventario Demanda"] < 0][
    [
        "Inventory Level",
        "Demand Forecast",
        "Units Sold",
        "Brecha Inventario Demanda"
    ]
]

riesgo.describe().round(2)

,Inventory Level,Demand Forecast,Units Sold,Brecha Inventario Demanda
count,2585.00,2585.00,2585.00,2585.00
mean,193.35,200.04,187.05,-6.69
std,123.26,123.24,123.20,4.84
min,50.00,50.86,33.00,-19.97
25%,90.00,97.77,84.00,-10.08
50%,156.00,162.79,149.00,-5.80
75%,276.00,283.11,271.00,-2.62
max,500.00,518.55,499.00,-0.01


In [19]:
ventas_superan_inventario = (
    riesgo["Units Sold"] > riesgo["Inventory Level"]
).sum()

ventas_superan_inventario

porcentaje_ventas_superan = (
    ventas_superan_inventario / len(riesgo) * 100
)

round(porcentaje_ventas_superan, 2)

np.float64(0.0)

In [20]:
porcentaje_ventas_superan = (
    ventas_superan_inventario / len(riesgo) * 100
)

round(porcentaje_ventas_superan, 2)

np.float64(0.0)

In [21]:
riesgo_tienda = (
    df[df["Brecha Inventario Demanda"] < 0]
    .groupby("Store ID")
    .size()
    .sort_values(ascending=False)
)

riesgo_tienda

Store ID
S004    547
S005    527
S002    523
S003    506
S001    482
dtype: int64

In [22]:
porcentaje_riesgo_tienda = (
    df[df["Brecha Inventario Demanda"] < 0]
    .groupby("Store ID")
    .size()
    .div(df.groupby("Store ID").size())
    .mul(100)
    .round(2)
)

porcentaje_riesgo_tienda

Store ID
S001    3.30
S002    3.58
S003    3.46
S004    3.74
S005    3.60
dtype: float64

In [23]:
riesgo_producto = (
    df[df["Brecha Inventario Demanda"] < 0]
    .groupby("Product ID")
    .size()
    .sort_values(ascending=False)
)

riesgo_producto

Product ID
P0004    149
P0019    146
P0013    143
P0010    141
P0009    140
P0008    138
P0020    130
P0003    129
P0002    128
P0011    127
P0007    127
P0006    125
P0001    124
P0018    123
P0016    122
P0005    122
P0017    121
P0015    120
P0014    116
P0012    114
dtype: int64

In [24]:
porcentaje_riesgo_producto = (
    df[df["Brecha Inventario Demanda"] < 0]
    .groupby("Product ID")
    .size()
    .div(df.groupby("Product ID").size())
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

porcentaje_riesgo_producto

Product ID
P0004    4.08
P0019    3.99
P0013    3.91
P0010    3.86
P0009    3.83
P0008    3.78
P0020    3.56
P0003    3.53
P0002    3.50
P0011    3.47
P0007    3.47
P0006    3.42
P0001    3.39
P0018    3.37
P0016    3.34
P0005    3.34
P0017    3.31
P0015    3.28
P0014    3.17
P0012    3.12
dtype: float64

In [25]:
df["Error Forecast"] = (
    df["Demand Forecast"] - df["Units Sold"]
)

df["Error Forecast"].describe().round(2)

count    73100.00
mean         5.06
std          8.62
min        -10.00
25%         -2.35
50%          4.99
75%         12.51
max         20.00
Name: Error Forecast, dtype: float64

In [26]:
sobreestimacion = (
    (df["Error Forecast"] > 0).sum()
)

subestimacion = (
    (df["Error Forecast"] < 0).sum()
)

igual = (
    (df["Error Forecast"] == 0).sum()
)

print("sobreestimación:", sobreestimacion)
print("subestimación:", subestimacion)
print("igual:", igual)

sobreestimación: 48783
subestimación: 24160
igual: 157


In [27]:
print(
    "sobreestimación %:",
    round(sobreestimacion / len(df) * 100, 2)
)

print(
    "subestimación %:",
    round(subestimacion / len(df) * 100, 2)
)

print(
    "igual %:",
    round(igual / len(df) * 100, 2)
)

sobreestimación %: 66.73
subestimación %: 33.05
igual %: 0.21


In [28]:
(df["Units Sold"] == 0).sum()


np.int64(360)

In [29]:
df_mape = df[df["Units Sold"] > 0].copy()

df_mape["Error Porcentual"] = (
    abs(df_mape["Demand Forecast"] - df_mape["Units Sold"])
    / df_mape["Units Sold"]
    * 100
)

df_mape["Error Porcentual"].describe().round(2)

count    72740.00
mean        23.29
std         86.96
min          0.00
25%          2.75
50%          6.52
75%         16.19
max       1996.00
Name: Error Porcentual, dtype: float64

In [30]:
df_mape_10 = df_mape[df_mape["Units Sold"] >= 10].copy()

df_mape_10["Error Porcentual"].describe().round(2)

count    69412.00
mean        12.56
std         18.81
min          0.00
25%          2.62
50%          6.07
75%         14.06
max        199.60
Name: Error Porcentual, dtype: float64

In [31]:
df_mape_10["Error Forecast"] = (
    df_mape_10["Demand Forecast"] - df_mape_10["Units Sold"]
)

df_mape_10["Tipo Error"] = df_mape_10["Error Forecast"].apply(
    lambda x: "sobreestimacion" if x > 0
    else "subestimacion" if x < 0
    else "sin error"
)

df_mape_10["Tipo Error"].value_counts()

Tipo Error
sobreestimacion    46318
subestimacion      23070
sin error             24
Name: count, dtype: int64

In [32]:
error_por_tipo = (
    df_mape_10
    .groupby("Tipo Error")["Error Forecast"]
    .agg(["count", "mean", "median"])
    .round(2)
)

error_por_tipo


,count,mean,median
Tipo Error,,,
sin error,24,0.00,0.00
sobreestimacion,46318,10.00,10.01
subestimacion,23070,-4.97,-4.95


In [33]:
df["Cobertura de Inventario"] = (
    df["Inventory Level"] / df["Demand Forecast"].replace(0, pd.NA)
)

df["Cobertura de Inventario"].describe().round(2)


count     72427.00
unique    70904.00
top           2.08
freq          8.00
Name: Cobertura de Inventario, dtype: float64

In [34]:
df["Cobertura de Inventario"] = pd.to_numeric(
    df["Cobertura de Inventario"],
    errors="coerce"
)

df["Cobertura de Inventario"].describe().round(2)

count    72427.00
mean         6.69
std        130.41
min          0.73
25%          1.29
50%          1.90
75%          3.57
max      22300.00
Name: Cobertura de Inventario, dtype: float64

In [35]:
df["Cobertura de Inventario"].quantile(
    [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).round(2)

0.01     0.94
0.05     1.02
0.10     1.08
0.25     1.29
0.50     1.90
0.75     3.57
0.90     7.76
0.95    13.24
0.99    45.53
Name: Cobertura de Inventario, dtype: float64

In [36]:
df["Nivel Cobertura"] = pd.cut(
    df["Cobertura de Inventario"],
    bins=[0, 1, 2, 4, 10, float("inf")],
    labels=[
        "menor a 1",
        "1 a 2",
        "2 a 4",
        "4 a 10",
        "mayor a 10"
    ],
    right=False
)

df["Nivel Cobertura"].value_counts().sort_index()

Nivel Cobertura
menor a 1      2585
1 a 2         35469
2 a 4         18475
4 a 10        10676
mayor a 10     5222
Name: count, dtype: int64

In [37]:
analisis_cobertura = (
    df.groupby("Nivel Cobertura", observed=True)
    .agg(
        inventario_promedio=("Inventory Level", "mean"),
        unidades_vendidas_promedio=("Units Sold", "mean"),
        unidades_ordenadas_promedio=("Units Ordered", "mean"),
        forecast_promedio=("Demand Forecast", "mean"),
        ventas_estimadas_promedio=("Estimated Sales", "mean")
    )
    .round(2)
)

analisis_cobertura

,inventario_promedio,unidades_vendidas_promedio,unidades_ordenadas_promedio,forecast_promedio,ventas_estimadas_promedio
Nivel Cobertura,,,,,
menor a 1,193.35,187.05,110.51,200.04,9326.28
1 a 2,275.80,200.96,109.80,206.14,9962.27
2 a 4,274.78,97.94,110.18,103.05,4857.85
4 a 10,281.84,44.51,110.13,49.25,2226.73
mayor a 10,299.08,14.85,110.46,16.72,745.22


In [38]:
alta_cobertura = (
    df[df["Nivel Cobertura"] == "mayor a 10"]
    .groupby(["Category", "Product ID"])
    .agg(
        registros=("Product ID", "size"),
        inventario_promedio=("Inventory Level", "mean"),
        unidades_vendidas_promedio=("Units Sold", "mean"),
        forecast_promedio=("Demand Forecast", "mean"),
        ventas_estimadas_promedio=("Estimated Sales", "mean")
    )
    .sort_values("registros", ascending=False)
    .round(2)
)

alta_cobertura.head(20)

registros  inventario_promedio  \
Category    Product ID                                   
Groceries   P0010              74               329.80   
            P0009              71               292.66   
Toys        P0003              66               275.76   
Groceries   P0012              65               308.57   
Toys        P0019              65               321.65   
Electronics P0016              64               273.55   
Groceries   P0003              64               285.73   
Toys        P0008              64               319.64   
Clothing    P0012              64               284.31   
Groceries   P0013              63               317.29   
Clothing    P0015              63               261.59   
Furniture   P0010              62               304.84   
Electronics P0004              62               323.45   
Furniture   P0002              61               298.89   
Clothing    P0018              60               313.93   
Toys        P0010              60               302.80   
Furniture   P0009              60               307.45   
Clothing    P0019              59               316.24   
Toys        P0004              59               291.66   
Groceries   P0017              59               313.08   

                        unidades_vendidas_promedio  forecast_promedio  \
Category    Product ID                                                  
Groceries   P0010                            17.14              18.31   
            P0009                            15.07              18.59   
Toys        P0003                            13.47              16.23   
Groceries   P0012                            15.11              16.89   
Toys        P0019                            17.45              18.67   
Electronics P0016                            14.95              13.38   
Groceries   P0003                            15.19              15.53   
Toys        P0008                            15.58              19.29   
Clothing    P0012                            15.19              16.14   
Groceries   P0013                            16.94              18.41   
Clothing    P0015                            13.40              14.94   
Furniture   P0010                            15.82              18.08   
Electronics P0004                            17.18              19.27   
Furniture   P0002                            13.59              16.67   
Clothing    P0018                            16.52              18.03   
Toys        P0010                            15.42              16.97   
Furniture   P0009                            16.02              16.94   
Clothing    P0019                            15.59              17.77   
Toys        P0004                            15.27              17.47   
Groceries   P0017                            15.71              17.68   

                        ventas_estimadas_promedio  
Category    Product ID                             
Groceries   P0010                          949.12  
            P0009                          772.39  
Toys        P0003                          759.25  
Groceries   P0012                          648.52  
Toys        P0019                          947.28  
Electronics P0016                          782.17  
Groceries   P0003                          741.66  
Toys        P0008                          790.37  
Clothing    P0012                          765.62  
Groceries   P0013                          921.08  
Clothing    P0015                          745.28  
Furniture   P0010                          804.99  
Electronics P0004                          931.03  
Furniture   P0002                          776.97  
Clothing    P0018                          818.84  
Toys        P0010                          756.64  
Furniture   P0009                          819.31  
Clothing    P0019                          775.61  
Toys        P0004                          777.46  
Groceries   P0017                          796.0

In [39]:
id="7p3kqm"
cobertura_producto = (
    df.groupby(["Category", "Product ID"])
    .agg(
        registros_totales=("Product ID", "size"),
        registros_alta_cobertura=(
            "Nivel Cobertura",
            lambda x: (x == "mayor a 10").sum()
        ),
        inventario_promedio=("Inventory Level", "mean"),
        unidades_vendidas_promedio=("Units Sold", "mean"),
        forecast_promedio=("Demand Forecast", "mean")
    )
)

cobertura_producto["porcentaje_alta_cobertura"] = (
    cobertura_producto["registros_alta_cobertura"]
    / cobertura_producto["registros_totales"]
    * 100
)

cobertura_producto = (
    cobertura_producto
    .sort_values("porcentaje_alta_cobertura", ascending=False)
    .round(2)
)

cobertura_producto.head(20)

registros_totales  registros_alta_cobertura  \
Category    Product ID                                                
Groceries   P0010                     728                        74   
            P0009                     721                        71   
Electronics P0016                     728                        64   
Toys        P0008                     728                        64   
            P0019                     743                        65   
            P0003                     755                        66   
Groceries   P0012                     745                        65   
Clothing    P0015                     726                        63   
Furniture   P0010                     723                        62   
Groceries   P0003                     759                        64   
Electronics P0004                     747                        62   
Clothing    P0012                     772                        64   
Groceries   P0013                     769                        63   
            P0007                     696                        57   
            P0017                     726                        59   
Furniture   P0002                     753                        61   
Electronics P0001                     700                        56   
Clothing    P0018                     752                        60   
Toys        P0010                     753                        60   
            P0004                     745                        59   

                        inventario_promedio  unidades_vendidas_promedio  \
Category    Product ID                                                    
Groceries   P0010                    280.26                      133.31   
            P0009                    276.98                      137.24   
Electronics P0016                    276.51                      138.76   
Toys        P0008                    271.43                      129.93   
            P0019                    278.94                      136.93   
            P0003                    273.67                      134.81   
Groceries   P0012                    271.28                      128.28   
Clothing    P0015                    272.37                      138.70   
Furniture   P0010                    271.73                      139.69   
Groceries   P0003                    279.10                      137.72   
Electronics P0004                    269.68                      131.95   
Clothing    P0012                    267.78                      127.41   
Groceries   P0013                    281.31                      138.81   
            P0007                    277.29                      142.45   
            P0017                    279.40                      132.75   
Furniture   P0002                    280.02                      131.46   
Electronics P0001                    270.51                      133.71   
Clothing    P0018                    280.03                      141.99   
Toys        P0010                    272.72                      132.75   
            P0004                    268.96                      133.48   

                        forecast_promedio  porcentaje_alta_cobertura  
Category    Product ID                                                
Groceries   P0010                  137.91                      10.16  
            P0009                  142.42                       9.85  
Electronics P0016                  143.52                       8.79  
Toys        P0008                  135.10                       8.79  
            P0019                  141.50                       8.75  
            P0003                  140.25                       8.74  
Groceries   P0012                  133.53                       8.72  
Clothing    P0015                  143.94                       8.68  
Furniture   P0010                  144.69                       8.58  
Groceries   P0003            

In [40]:
cobertura_categoria = (
    df.groupby("Category")
    .agg(
        registros_totales=("Category", "size"),
        registros_alta_cobertura=(
            "Nivel Cobertura",
            lambda x: (x == "mayor a 10").sum()
        ),
        inventario_promedio=("Inventory Level", "mean"),
        unidades_vendidas_promedio=("Units Sold", "mean"),
        forecast_promedio=("Demand Forecast", "mean"),
        ventas_estimadas_promedio=("Estimated Sales", "mean")
    )
)

cobertura_categoria["porcentaje_alta_cobertura"] = (
    cobertura_categoria["registros_alta_cobertura"]
    / cobertura_categoria["registros_totales"]
    * 100
)

cobertura_categoria.sort_values(
    "porcentaje_alta_cobertura",
    ascending=False
).round(2)

,registros_totales,registros_alta_cobertura,inventario_promedio,unidades_vendidas_promedio,forecast_promedio,ventas_estimadas_promedio,porcentaje_alta_cobertura
Category,,,,,,,
Groceries,14611,1089,275.76,136.92,141.90,6840.67,7.45
Electronics,14521,1038,272.51,135.01,140.04,6706.53,7.15
Furniture,14699,1048,275.82,137.77,142.86,6818.89,7.13
Clothing,14626,1024,274.60,136.69,141.78,6746.67,7.00
Toys,14643,1023,273.65,135.93,141.05,6742.42,6.99


In [41]:
impacto_cobertura = (
    df.groupby("Nivel Cobertura", observed=True)
    .agg(
        registros=("Product ID", "size"),
        inventario_total=("Inventory Level", "sum"),
        unidades_vendidas_totales=("Units Sold", "sum"),
        ventas_estimadas_totales=("Estimated Sales", "sum")
    )
)

impacto_cobertura["porcentaje_inventario"] = (
    impacto_cobertura["inventario_total"]
    / df["Inventory Level"].sum()
    * 100
)

impacto_cobertura["porcentaje_ventas"] = (
    impacto_cobertura["ventas_estimadas_totales"]
    / df["Estimated Sales"].sum()
    * 100
)

impacto_cobertura.round(2)


,registros,inventario_total,unidades_vendidas_totales,ventas_estimadas_totales,porcentaje_inventario,porcentaje_ventas
Nivel Cobertura,,,,,,
menor a 1,2585,499822,483518,2.410844e+07,2.49,4.87
1 a 2,35469,9782395,7127896,3.533516e+08,48.76,71.39
2 a 4,18475,5076478,1809407,8.974880e+07,25.30,18.13
4 a 10,10676,3008892,475242,2.377259e+07,15.00,4.80
mayor a 10,5222,1561795,77559,3.891517e+06,7.78,0.79


In [42]:
df["Nivel Rotacion"] = pd.qcut(
    df["Units Sold"],
    q=4,
    labels=[
        "baja",
        "media-baja",
        "media-alta",
        "alta"
    ]
)

rotacion_cobertura = (
    df.groupby("Nivel Rotacion", observed=True)
    .agg(
        registros=("Product ID", "size"),
        inventario_promedio=("Inventory Level", "mean"),
        unidades_vendidas_promedio=("Units Sold", "mean"),
        forecast_promedio=("Demand Forecast", "mean"),
        cobertura_mediana=("Cobertura de Inventario", "median"),
        ventas_estimadas_promedio=("Estimated Sales", "mean")
    )
    .round(2)
)

rotacion_cobertura

,registros,inventario_promedio,unidades_vendidas_promedio,forecast_promedio,cobertura_mediana,ventas_estimadas_promedio
Nivel Rotacion,,,,,,
baja,18642,196.60,24.55,29.65,5.92,1219.51
media-baja,17916,224.14,76.66,81.72,2.39,3793.23
media-alta,18394,291.52,151.21,156.29,1.76,7494.93
alta,18148,386.87,295.53,300.53,1.24,14680.18
